<a href="https://colab.research.google.com/github/Melissa-Etes/Sprint2_grupo54/blob/main/EC_Sprint2_HospiDataSUS_Clusterizacao_Grupo54.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clusterização de Municípios — HospiData SUS - GRUPO 54
Análise de padrões de sobrecarga hospitalar por município, usando K-Means.

## 1. Importações e carga dos dados

In [32]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

SEED = 1224
np.random.seed(SEED)

##### Carregando Dados do GitHub

In [10]:
# Carrega os dados tratados (exportados do Oracle via ETL) direto do GitHub
dados = pd.read_csv('https://raw.githubusercontent.com/Melissa-Etes/Sprint2_grupo54/main/data/internacoes_tratado.csv')
dados.head()

,COD_MUNICIPIO,MUNICIPIO,QTD_INTERNACOES,PERMANENCIA_MEDIA,TAXA_MORTALIDADE_PCT,VALOR_MEDIO_INTERNACAO,QTD_ESTAB_HOSPITALARES,POPULACAO,INTERNACOES_POR_MIL_HAB
0,353760,Peruíbe,263,5.55,11.03,1472.34,0,68352,3.85
1,350190,Amparo,336,6.09,10.71,1931.03,0,68008,4.94
2,351020,Capão Bonito,261,3.50,10.73,1310.42,0,46337,5.63
3,350660,Biritiba Mirim,136,5.42,11.03,1676.72,0,29683,4.58
4,353600,Parapuã,57,7.28,8.77,1149.48,0,10580,5.39


,COD_MUNICIPIO,MUNICIPIO,QTD_INTERNACOES,PERMANENCIA_MEDIA,TAXA_MORTALIDADE_PCT,VALOR_MEDIO_INTERNACAO,QTD_ESTAB_HOSPITALARES,POPULACAO,INTERNACOES_POR_MIL_HAB
0,353760,Peruíbe,263,5.55,11.03,1472.34,0,68352,3.85
1,350190,Amparo,336,6.09,10.71,1931.03,0,68008,4.94
2,351020,Capão Bonito,261,3.50,10.73,1310.42,0,46337,5.63
3,350660,Biritiba Mirim,136,5.42,11.03,1676.72,0,29683,4.58
4,353600,Parapuã,57,7.28,8.77,1149.48,0,10580,5.39


##### Conferir os dados

In [11]:
dados.shape
dados.isna().sum()

,0
COD_MUNICIPIO,0
MUNICIPIO,0
QTD_INTERNACOES,0
PERMANENCIA_MEDIA,0
TAXA_MORTALIDADE_PCT,0
VALOR_MEDIO_INTERNACAO,0
QTD_ESTAB_HOSPITALARES,0
POPULACAO,0
INTERNACOES_POR_MIL_HAB,0


,0
COD_MUNICIPIO,0
MUNICIPIO,0
QTD_INTERNACOES,0
PERMANENCIA_MEDIA,0
TAXA_MORTALIDADE_PCT,0
VALOR_MEDIO_INTERNACAO,0
QTD_ESTAB_HOSPITALARES,0
POPULACAO,0
INTERNACOES_POR_MIL_HAB,0


In [12]:
dados = dados.dropna(subset=['POPULACAO'])
dados['QTD_ESTAB_HOSPITALARES'] = dados['QTD_ESTAB_HOSPITALARES'].fillna(0)

## 2. Correção de viés: Razão Observado/Esperado

A taxa bruta de internações por mil habitantes distorce municípios pequenos (qualquer número de internações vira uma taxa desproporcional). Corrigimos calculando quantas internações seriam **esperadas** para cada município, dado o tamanho da população, e comparando com o valor **observado**.

In [33]:
# Taxa média geral de internação (ponderada pela população total)
taxa_media_geral = dados['QTD_INTERNACOES'].sum() / dados['POPULACAO'].sum() * 1000

# Número esperado de internações para cada município
dados['internacoes_esperadas'] = taxa_media_geral * dados['POPULACAO'] / 1000

# Razão Observado/Esperado: >1 = acima da média (proporcional à população), <1 = abaixo
dados['razao_obs_esperado'] = dados['QTD_INTERNACOES'] / dados['internacoes_esperadas']

dados[['MUNICIPIO', 'POPULACAO', 'QTD_INTERNACOES', 'internacoes_esperadas', 'razao_obs_esperado']] \
    .sort_values('razao_obs_esperado', ascending=False).head(10)

,MUNICIPIO,POPULACAO,QTD_INTERNACOES,internacoes_esperadas,razao_obs_esperado
439,Divinolândia,11158,266,49.609130,5.361916
382,Jaci,7613,180,33.847850,5.317915
279,Lucianópolis,2372,29,10.546053,2.749844
125,Casa Branca,28083,340,124.858685,2.723079
153,Ibirá,11690,139,51.974434,2.674392
456,Cajuru,23830,282,105.949594,2.661643
612,Vitória Brasil,1794,21,7.976230,2.632823
358,Nantes,2660,31,11.826518,2.621228
112,Emilianópolis,3014,35,13.400423,2.611858
503,Uru,1387,16,6.166684,2.594587


In [34]:
# Conferindo os municípios que antes apareciam como outliers isolados
dados[dados['MUNICIPIO'].isin(['Jaci', 'Divinolândia'])][
    ['MUNICIPIO', 'POPULACAO', 'QTD_INTERNACOES', 'internacoes_esperadas', 'razao_obs_esperado']
]

,MUNICIPIO,POPULACAO,QTD_INTERNACOES,internacoes_esperadas,razao_obs_esperado
382,Jaci,7613,180,33.84785,5.317915
439,Divinolândia,11158,266,49.60913,5.361916


## 3. Seleção de features

**Importante:** não incluímos `QTD_INTERNACOES` (número bruto) nem `INTERNACOES_POR_MIL_HAB` — ambas ainda carregam o viés de tamanho populacional. Usamos `razao_obs_esperado` no lugar, que já é a versão normalizada dessa informação.

In [35]:
features = dados[['PERMANENCIA_MEDIA', 'TAXA_MORTALIDADE_PCT',
                   'QTD_ESTAB_HOSPITALARES', 'razao_obs_esperado']]
features.head()

,PERMANENCIA_MEDIA,TAXA_MORTALIDADE_PCT,QTD_ESTAB_HOSPITALARES,razao_obs_esperado
0,5.55,11.03,0,0.865425
1,6.09,10.71,0,1.111230
2,3.50,10.73,0,1.266885
3,5.42,11.03,0,1.030519
4,7.28,8.77,0,1.211753


## 4. Pipeline de padronização + PCA

In [36]:
pca_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('PCA', PCA(n_components=2, random_state=SEED))
])

embedding_pca = pca_pipeline.fit_transform(features)
projection = pd.DataFrame(columns=['x', 'y'], data=embedding_pca)
projection.head()

,x,y
0,-0.695129,0.785925
1,-0.141817,0.863757
2,-0.669666,0.055906
3,-0.483150,0.721974
4,0.492785,0.907041


In [37]:
print("Variância explicada (2D):", pca_pipeline.named_steps['PCA'].explained_variance_ratio_.sum())

Variância explicada (2D): 0.7576131069040257


## 5. Método do Cotovelo — escolha do número de clusters

In [38]:
inercia = []
for k in range(1, 10):
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    km.fit(embedding_pca)
    inercia.append(km.inertia_)

fig_cotovelo = px.line(
    x=list(range(1, 10)), y=inercia, markers=True,
    title='Método do Cotovelo',
    labels={'x': 'Número de clusters (k)', 'y': 'Inércia'}
)
fig_cotovelo.show()

In [39]:
# Salva o gráfico como imagem para usar no PPT/README
# (se der erro de kaleido, rode: !pip install kaleido==0.2.1  e reinicie a sessão)
fig_cotovelo.write_image("metodo_cotovelo.png")

## 6. Aplicação do K-Means

In [40]:
K_ESCOLHIDO = 3  # ajuste conforme o gráfico do cotovelo

kmeans = KMeans(n_clusters=K_ESCOLHIDO, random_state=SEED, n_init=10)
projection['cluster'] = kmeans.fit_predict(embedding_pca).astype(str)
projection['municipio'] = dados['MUNICIPIO'].values
dados['cluster'] = projection['cluster'].values
projection.head()

,x,y,cluster,municipio
0,-0.695129,0.785925,0,Peruíbe
1,-0.141817,0.863757,0,Amparo
2,-0.669666,0.055906,0,Capão Bonito
3,-0.483150,0.721974,0,Biritiba Mirim
4,0.492785,0.907041,0,Parapuã


## 7. Visualização dos clusters

In [41]:
fig_clusters = px.scatter(
    projection, x='x', y='y', color='cluster',
    hover_data=['municipio'],
    title='Clusters de Municípios (Métrica Corrigida - Razão Observado/Esperado)'
)
fig_clusters.update_traces(marker=dict(size=8, opacity=0.9, line=dict(width=0.5, color='#121212')))
fig_clusters.show()

In [42]:
fig_clusters.write_image("clusters_municipios.png")

## 8. Interpretação dos clusters

In [43]:
dados.groupby('cluster')[
    ['PERMANENCIA_MEDIA', 'TAXA_MORTALIDADE_PCT', 'QTD_ESTAB_HOSPITALARES', 'razao_obs_esperado']
].mean()

,PERMANENCIA_MEDIA,TAXA_MORTALIDADE_PCT,QTD_ESTAB_HOSPITALARES,razao_obs_esperado
cluster,,,,
0,5.594929,10.774574,0.0,1.050639
1,4.627895,4.506870,0.0,1.366749
2,21.180000,1.305000,0.0,5.339916


In [44]:
# Quantos municípios em cada cluster
dados['cluster'].value_counts()

,count
cluster,
1,361
0,282
2,2


## 9. Investigação de outliers

Se algum cluster tiver poucos municípios muito isolados, investigamos individualmente.

In [45]:
dados[dados['cluster'] == '1'][
    ['MUNICIPIO', 'POPULACAO', 'QTD_INTERNACOES', 'PERMANENCIA_MEDIA',
     'TAXA_MORTALIDADE_PCT', 'QTD_ESTAB_HOSPITALARES', 'razao_obs_esperado']
]

,MUNICIPIO,POPULACAO,QTD_INTERNACOES,PERMANENCIA_MEDIA,TAXA_MORTALIDADE_PCT,QTD_ESTAB_HOSPITALARES,razao_obs_esperado
5,Palmares Paulista,9650,42,5.17,0.00,0,0.978919
6,Três Fronteiras,6804,37,5.41,0.00,0,1.223101
10,Jaguariúna,59347,292,5.75,4.45,0,1.106646
12,Paulicéia,7955,38,6.37,5.26,0,1.074405
16,Rio Grande da Serra,44170,150,5.41,2.67,0,0.763816
...,...,...,...,...,...,...,...
636,José Bonifácio,36633,242,4.39,9.50,0,1.485825
638,Martinópolis,24881,200,4.89,5.50,0,1.807952
639,Tambaú,21435,120,5.85,4.17,0,1.259165
641,João Ramalho,4371,22,5.27,0.00,0,1.132053
